# Visual-inertial encoder training (Colab)

Trains the triage visual encoder variants on pre-collected rollout data.
The Rust sim is not needed here — data is uploaded.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 or better)
2. Have these files ready locally:
   - `target/venc-data/train-paired/rollouts.pt`
   - `target/venc-data/val-paired/rollouts.pt`
   - `target/venc-data/real/frames.pt`
3. Repo access: either make the repo public, use a GitHub token, or upload a zip of the `rl/` directory.

In [ ]:
!nvidia-smi -L
import torch
print(torch.__version__, torch.cuda.is_available())

## 1. Get the code

Option A: clone the repo (needs a token if private). Option B: upload a zip of `rl/`.

In [ ]:
import os

# Option A: clone (set TOKEN for private repos, or leave empty if public)
TOKEN = ""  # GitHub personal access token, or ""
REPO = "github.com/jarenm1/triage.git"

if not os.path.exists("triage"):
    if TOKEN:
        os.system(f"git clone --depth 1 https://{TOKEN}@{REPO} triage")
    else:
        os.system(f"git clone --depth 1 https://{REPO} triage")

os.chdir("triage")
print(os.listdir("rl"))

In [ ]:
# Option B (alternative): upload rl.zip containing the rl/ directory
# Run locally first:  cd triage && zip -r rl.zip rl/
# from google.colab import files
# uploaded = files.upload()  # pick rl.zip
# !unzip -o rl.zip -d triage
# import os; os.chdir('triage')

## 2. Install deps

In [ ]:
!pip install -q omegaconf huggingface_hub
import sys
sys.path.insert(0, ".")
from rl.visual_encoder import VisualEncoder, VisualEncoderConfig
enc = VisualEncoder()
print("encoder ok, params:", sum(p.numel() for p in enc.parameters()))

## 3. Upload data

Three files, ~5GB total — Drive required. Frames are JPEG-packed
(decode on batch in the trainer).

- `train-256-rollouts.pt` — sim rollouts, 256×192, paired appearances
- `val-256-rollouts.pt` — held-out sim rollouts
- `real-256.pt` — 23.5k real FPV frames (19 clips)


In [ ]:
import os
from pathlib import Path

DATA = Path("target/venc-data")
(DATA / "train-256").mkdir(parents=True, exist_ok=True)
(DATA / "val-256").mkdir(parents=True, exist_ok=True)
(DATA / "real").mkdir(parents=True, exist_ok=True)

# Google Drive: put the three files in Drive/triage-data/ then:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/triage-data/train-256-rollouts.pt target/venc-data/train-256/rollouts.pt
!cp /content/drive/MyDrive/triage-data/val-256-rollouts.pt target/venc-data/val-256/rollouts.pt
!cp /content/drive/MyDrive/triage-data/real-256.pt target/venc-data/real/real-256.pt

for p in [DATA/"train-256/rollouts.pt", DATA/"val-256/rollouts.pt", DATA/"real/real-256.pt"]:
    print(p, "exists:", p.exists())


## 4. Train

Variants: `a`–`h` (core ablation), `i` (+appearance consistency), `j` (+DINOv2 distill),
`k` (+domain adversarial on real frames), `l` (k + teacher distill).

Teacher options: `dinov2` (open), `lingbot` (ViT-L/16 dense-perception, open), `dinov3` (gated, needs `--teacher-ckpt`).

Recommended: variant `l` with `--teacher lingbot` — the full sim↔real alignment stack.

In [ ]:
import os
VARIANT = "m"          # distill on sim+real, disc eval-only
TEACHER = "lingbot"
TEACHER_VARIANT = "base"   # ViT-B: fits T4 alongside the student
STEPS = 4000
BATCH = 16

cmd = (
    f"python -m rl.venc_train --variant {VARIANT} "
    f"--data target/venc-data/train-256 "
    f"--val target/venc-data/val-256 "
    f"--real target/venc-data/real/real-256.pt "
    f"--teacher {TEACHER} --teacher-variant {TEACHER_VARIANT} "
    f"--batch {BATCH} --steps {STEPS} "
    f"--output target/venc-results/{VARIANT}-256-colab.json"
)
print(cmd)
os.system(cmd)


## 5. Results

In [ ]:
import json
r = json.load(open(f"target/venc-results/{VARIANT}-colab.json"))
print(json.dumps({"bench": r["bench"], "final": r["final"]}, indent=2))

In [ ]:
# Occlusion retention eval
os.system(
    f"python -m rl.venc_eval "
    f"--checkpoint target/venc-results/{VARIANT}-colab.pt "
    f"--variant {VARIANT} --val target/venc-data/val-paired"
)

In [ ]:
# Download results + checkpoint
# from google.colab import files
# files.download(f"target/venc-results/{VARIANT}-colab.json")
# files.download(f"target/venc-results/{VARIANT}-colab.pt")